# Pre-encoded BEHAVIOR Dataset — Quick Validation

Validates MDS shards on a **local network volume** (RunPod).
All heavy checks use sample-based shard reads so the full notebook runs in **~2–3 minutes**
against a multi-TB dataset.

**What these tests cannot guarantee:**
- Correctness for rows / episodes in unsampled shards (Tests 2–4)
- Pixel quality or semantic meaningfulness of V-JEPA2 embeddings
- Action/state correctness for episodes outside the N=5 spot-check (Tests 5–6)

In [ ]:
!pip install pandas
!pip install decord
!pip install cython
!pip install streaming
!pip install mosaicml-streaming
!pip install transformers==4.35.2
!pip install pyarrow
# pip install mosaicml-streaming pandas numpy pyarrow decord huggingface_hub hf_transfer
import json, os, tempfile
from collections import defaultdict
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
from decord import VideoReader, cpu
from streaming import StreamingDataset

# ── Paths — adjust to your RunPod environment ─────────────────────────────────
LOCAL_SHARD_DIR = Path("/workspace/shards_256px_vit_16_g")
BASE_DATASET    = Path("/workspace/base_dataset")
MANIFEST_PATH   = BASE_DATASET / "manifest.json"

# ── HuggingFace upload target ──────────────────────────────────────────────────
HF_REPO_ID     = "quastAI/behavior2025-vjepa2-vitg-130h-demo-embeddings"
HF_PATH_PREFIX = "shards_256px_vit_16_g"

# ── Constants — must match configs/behavior-vjepa2-vitg16-256px-16f.yaml ──────
TARGET_FPS      = 5
ACTION_DIM      = 23
STATE_DIM       = 133
NATIVE_FPS      = 30
FSTP_NOM        = ceil(NATIVE_FPS / TARGET_FPS)   # = 6
ACTIONS_PER_ROW = FSTP_NOM * ACTION_DIM            # = 138
CAMERA_VIEW     = "multi"
views           = ["head", "left_wrist", "right_wrist"]

# ── Sampling knobs ─────────────────────────────────────────────────────────────
N_SHARD_SAMPLE = 5   # shards read fully (spread: first, 25%, 50%, 75%, last)
N_SPOT_CHECK   = 5   # episodes cross-checked against raw parquet
RNG_SEED       = 42

# ── Proprioceptive slices (must match BehaviorVideoDataset.PROPRIO_SLICES) ────
PROPRIO_SLICES = [
    np.s_[6:28], np.s_[34:56], np.s_[62:84], np.s_[84:112],
    np.s_[152:158], np.s_[186:197], np.s_[225:244], np.s_[253:256],
]

def extract_proprio(full_states: np.ndarray) -> np.ndarray:
    return np.concatenate([full_states[..., s] for s in PROPRIO_SLICES], axis=-1)

print(f"LOCAL_SHARD_DIR : {LOCAL_SHARD_DIR}")
print(f"FSTP_NOM={FSTP_NOM}  ACTIONS_PER_ROW={ACTIONS_PER_ROW}  STATE_DIM={STATE_DIM}")
print(f"views={views}  N_SHARD_SAMPLE={N_SHARD_SAMPLE}  N_SPOT_CHECK={N_SPOT_CHECK}")

---
## Test 1 — Index & Schema (instant)

Reads `index.json` only — no shard data touched.

**Guarantees:** The index file is valid; total row/shard counts are non-zero; every shard
carries all expected MDS columns; total dataset size is reported.

**Does NOT guarantee:** Numerical correctness of any values.

In [ ]:
assert LOCAL_SHARD_DIR.exists(), f"Shard directory not found: {LOCAL_SHARD_DIR}"

index_path = LOCAL_SHARD_DIR / "index.json"
assert index_path.exists(), "index.json missing — dataset may be incomplete"

with open(index_path) as f:
    index = json.load(f)

shards = index.get("shards", [])
assert len(shards) > 0, "index.json contains no shards"
total_rows = sum(s["samples"] for s in shards)
assert total_rows > 0, "Dataset is empty"

TOKEN_COLS = {f"tokens_{v}" for v in views}
EXPECTED_COLUMNS = TOKEN_COLS | {"actions", "states", "cam_rel_poses", "frame_index",
                                  "episode_idx", "sample_idx", "step_pos", "episode_len"}
for s in shards:
    cols = set(s.get("column_names", []))
    missing = EXPECTED_COLUMNS - cols
    assert not missing, f"Shard {s['raw_data']['basename']} missing columns: {missing}"

total_bytes = sum(s.get("raw_data", {}).get("size", 0) for s in shards)
print(f"Shards      : {len(shards):,}")
print(f"Total rows  : {total_rows:,}  (one row = one sampled frame)")
print(f"Total size  : {total_bytes / 1e12:.2f} TB  ({total_bytes / 1e9:.1f} GB)")
print("PASS Test 1 — index.json valid, schema OK across all shards.")

---
## Test 2 — Sample-Based Shape, Finiteness & Metadata (~1 min)

Opens **5 representative shards** (first, 25 %, 50 %, 75 %, last) via a temporary
sub-index, reads every row in those shards, and checks:

- All token arrays have a consistent 2-D `(tokens_per_frame, embed_dim)` shape and contain no NaN/Inf
- `actions` is `(ACTIONS_PER_ROW,)`, `states` is `(STATE_DIM,)`, `cam_rel_poses` is `(21,)`
- Collects `rows_meta` used by Tests 3–6

**Does NOT guarantee:** Correctness of rows in unsampled shards.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
n_shards = len(shards)

# Spread sample positions evenly; deduplicate for tiny datasets
sample_positions = sorted(set([
    0,
    n_shards // 4,
    n_shards // 2,
    3 * n_shards // 4,
    n_shards - 1,
][:N_SHARD_SAMPLE]))
sampled_shards = [shards[i] for i in sample_positions]
print(f"Sampling shard positions : {sample_positions} of {n_shards}")
print(f"Rows per sampled shard   : {[s['samples'] for s in sampled_shards]}")

# Build a temporary sub-index pointing only to the sampled shards (symlinks, no copies)
tmp_dir = tempfile.mkdtemp(prefix="behavior_val_")
for si in sampled_shards:
    basename = si["raw_data"]["basename"]
    src = LOCAL_SHARD_DIR / basename
    dst = Path(tmp_dir) / basename
    assert src.exists(), f"Shard file missing: {src}"
    os.symlink(src, dst)

with open(Path(tmp_dir) / "index.json", "w") as f:
    json.dump({"version": index.get("version", 2), "shards": sampled_shards}, f)

subset_ds = StreamingDataset(local=tmp_dir, shuffle=False)
n_subset = len(subset_ds)
print(f"\nSubset dataset: {n_subset} rows from {len(sampled_shards)} shards")

TOKEN_SHAPE = None
shape_failures, nan_tok_rows, nan_act_rows, nan_state_rows = [], [], [], []
rows_meta = []

for i in range(n_subset):
    row = subset_ds[i]
    tok = row[f"tokens_{views[0]}"]
    assert isinstance(tok, np.ndarray) and tok.ndim == 2, \
        f"row {i}: unexpected token type/shape for tokens_{views[0]}"

    if TOKEN_SHAPE is None:
        TOKEN_SHAPE = tok.shape
        print(f"  Detected token shape : {TOKEN_SHAPE}  (tokens_per_frame, embed_dim)")
        assert row["actions"].shape == (ACTIONS_PER_ROW,), \
            f"actions: expected ({ACTIONS_PER_ROW},), got {row['actions'].shape}"
        assert row["states"].shape == (STATE_DIM,), \
            f"states: expected ({STATE_DIM},), got {row['states'].shape}"
        assert row["cam_rel_poses"].shape == (21,), \
            f"cam_rel_poses: expected (21,), got {row['cam_rel_poses'].shape}"
    else:
        for v in views:
            t = row[f"tokens_{v}"]
            if t.shape != TOKEN_SHAPE:
                shape_failures.append(f"row {i} tokens_{v}: {t.shape} != {TOKEN_SHAPE}")

    # Finiteness: tokens
    for v in views:
        if not np.all(np.isfinite(row[f"tokens_{v}"])):
            nan_tok_rows.append((i, v))

    # Finiteness: actions and states
    if not np.all(np.isfinite(row["actions"])):
        nan_act_rows.append(i)
    if not np.all(np.isfinite(row["states"])):
        nan_state_rows.append(i)

    rows_meta.append({
        "episode_idx": int(row["episode_idx"]),
        "sample_idx":  int(row["sample_idx"]),
        "step_pos":    int(row["step_pos"]),
        "frame_index": int(row["frame_index"]),
        "episode_len": int(row["episode_len"]),
        "actions":     row["actions"],
        "states":      row["states"],
    })

if shape_failures:
    for f in shape_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(shape_failures)} token shape mismatches")

print(f"\nStreamed {n_subset:,} rows from {len(sampled_shards)} shards.")
print(f"Token shape : {TOKEN_SHAPE}")

all_ok = True
for label, bad in [("tokens NaN/Inf", nan_tok_rows),
                   ("actions NaN/Inf", nan_act_rows),
                   ("states NaN/Inf",  nan_state_rows)]:
    if bad:
        print(f"WARNING — {len(bad)} rows have {label}: e.g. {bad[:3]}")
        all_ok = False

if all_ok:
    print(f"All {n_subset:,} rows finite for tokens, actions, and states.")
print("PASS Test 2")

---
## Test 3 — Episode Completeness (within sampled shards)

For every episode **fully contained** in the sampled shards (observed rows span
`step_pos 0 … episode_len-1`), verifies that `step_pos` is exactly `0 … episode_len-1`
with no gaps and that `episode_len` is consistent across all rows.

Episodes split across shard boundaries are verified for monotonicity only.

**Does NOT guarantee:** Completeness of episodes in unsampled shards.

In [ ]:
episodes: dict = defaultdict(list)
for row in rows_meta:
    episodes[row["episode_idx"]].append(row)

n_full, n_partial, failures = 0, 0, []

for ep_id, ep_rows in episodes.items():
    stored_len = ep_rows[0]["episode_len"]

    # episode_len must agree across all rows of this episode
    if not all(r["episode_len"] == stored_len for r in ep_rows):
        failures.append(f"ep {ep_id}: inconsistent episode_len values")
        continue

    positions = sorted(r["step_pos"] for r in ep_rows)
    sorted_rows = sorted(ep_rows, key=lambda r: r["step_pos"])

    # First row (step_pos=0) must have frame_index=0
    first = sorted_rows[0]
    if first["step_pos"] == 0 and first["frame_index"] != 0:
        failures.append(
            f"ep {ep_id}: step_pos=0 has frame_index={first['frame_index']}, expected 0"
        )

    if len(ep_rows) == stored_len:
        # Full episode visible — check exact completeness
        n_full += 1
        if positions != list(range(stored_len)):
            failures.append(f"ep {ep_id}: step_pos gaps, expected 0..{stored_len-1}")
    else:
        # Partial episode (spans shard boundary) — check monotonicity
        n_partial += 1
        for a, b in zip(positions, positions[1:]):
            if b != a + 1:
                failures.append(f"ep {ep_id} (partial): step_pos gap {a}→{b}")

if failures:
    for f in failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} episode completeness failures")

print(f"Unique episodes in sample : {len(episodes)}")
print(f"  Fully contained         : {n_full}  (step_pos 0…n-1 + frame_index[0]==0 verified)")
print(f"  Partial (boundary)      : {n_partial}  (monotonicity + frame_index[0]==0 verified)")
print("PASS Test 3")

---
## Test 4 — Frame-Index Sampling Pattern (sampled shards)

For every episode in the sample, verifies that `frame_index` values are strictly
monotonically increasing and that consecutive steps are separated by exactly
`fstp = ceil(video_fps / TARGET_FPS)`.

Uses `decord` to read video FPS (head-camera, one open per episode).
Episodes whose video file is absent on this node are skipped.

**Does NOT guarantee:** Correct sampling for unsampled shards.

In [ ]:
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

_VIEW_IDX = {"head": 0, "left_wrist": 1, "right_wrist": 2}

def video_path(sample_idx, view="head"):
    ep = manifest["episodes"][sample_idx]
    vfiles = ep.get("video_files") or [ep.get("video_file", "")]
    idx = _VIEW_IDX.get(view, 0)
    name = Path(vfiles[min(idx, len(vfiles) - 1)]).name
    return BASE_DATASET / ep["task_name"] / "video" / view / name

def parquet_path(sample_idx):
    ep = manifest["episodes"][sample_idx]
    name = os.path.splitext(os.path.basename(ep["episode_file"]))[0]
    return BASE_DATASET / ep["task_name"] / "data" / f"{name}.parquet"

failures, skipped = [], 0
for ep_id, ep_rows in episodes.items():
    sorted_rows = sorted(ep_rows, key=lambda r: r["step_pos"])
    fis = [r["frame_index"] for r in sorted_rows]

    # Strict monotonicity (no video file needed)
    if any(fis[i] >= fis[i + 1] for i in range(len(fis) - 1)):
        failures.append(f"ep {ep_id}: frame_index not strictly increasing")
        continue

    vp = video_path(sorted_rows[0]["sample_idx"])
    if not vp.exists():
        skipped += 1
        continue  # video not on this node — skip step-size check

    vr = VideoReader(str(vp), num_threads=1, ctx=cpu(0))
    vfps = vr.get_avg_fps()
    del vr
    fstp = max(1, ceil(vfps / TARGET_FPS))

    steps = [fis[i + 1] - fis[i] for i in range(len(fis) - 1)]
    wrong = [s for s in steps if s != fstp]
    if wrong:
        failures.append(
            f"ep {ep_id} fstp={fstp}: {len(wrong)}/{len(steps)} wrong steps e.g. {wrong[:3]}"
        )

if failures:
    for f in failures: print("FAIL:", f)
    raise AssertionError(f"{len(failures)} sampling-pattern failures")

print(f"PASS Test 4 — {len(episodes)} episodes checked "
      f"({skipped} skipped: video not local), frame_index stride correct.")

---
## Test 5 — State Alignment vs. Base Parquet (spot-check)

For `N_SPOT_CHECK` randomly chosen episodes from the sample, reads the source parquet and
verifies `states == extract_proprio(parquet['observation.state'][frame_index])`.

**Does NOT guarantee:** Correctness for episodes outside the spot-check.

In [ ]:
rng2 = np.random.default_rng(RNG_SEED + 1)
available_ep_ids = list(episodes.keys())
check_ep_ids = rng2.choice(available_ep_ids,
                            size=min(N_SPOT_CHECK, len(available_ep_ids)),
                            replace=False)

state_failures = []
for ep_id in check_ep_ids:
    sorted_rows = sorted(episodes[ep_id], key=lambda r: r["step_pos"])
    pp = parquet_path(sorted_rows[0]["sample_idx"])
    if not pp.exists():
        print(f"  SKIP ep {ep_id}: parquet not found at {pp}")
        continue

    df = pd.read_parquet(pp, columns=["observation.state"])
    raw = np.asarray(df["observation.state"].to_list(), dtype=np.float32)  # (T, 256)

    vp = video_path(sorted_rows[0]["sample_idx"])
    if vp.exists():
        vr = VideoReader(str(vp), num_threads=1, ctx=cpu(0))
        vfps = vr.get_avg_fps()
        del vr
        fstp = max(1, ceil(vfps / TARGET_FPS))
    else:
        fstp = FSTP_NOM

    for row in sorted_rows:
        fi = row["frame_index"]
        expected = (np.zeros(STATE_DIM, dtype=np.float32)
                    if fi >= len(raw)
                    else extract_proprio(raw[fi : fi + 1]).flatten())
        if not np.allclose(expected, row["states"], rtol=1e-4, atol=1e-5):
            state_failures.append(
                f"ep {ep_id} step_pos={row['step_pos']} fi={fi}: "
                f"max_diff={np.max(np.abs(expected - row['states'])):.2e}"
            )
    print(f"  ep {ep_id}: {len(sorted_rows)} steps — OK")

if state_failures:
    for f in state_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(state_failures)} state alignment failures")
print("PASS Test 5 — states == extract_proprio(parquet[frame_index])")

---
## Test 6 — Action Aggregation vs. Base Parquet (spot-check)

For the same `N_SPOT_CHECK` episodes, verifies
`actions == flatten(raw_actions[frame_index : frame_index+fstp, :ACTION_DIM])`.

**Does NOT guarantee:** Correctness outside the spot-check.

In [ ]:
def reconstruct_action_chunk(raw_act, start, fstp, max_len):
    if start >= max_len:
        return np.zeros((fstp, ACTION_DIM), dtype=np.float32)
    end = min(start + fstp, max_len)
    chunk = raw_act[start:end]
    if len(chunk) == 0:
        return np.zeros((fstp, ACTION_DIM), dtype=np.float32)
    if len(chunk) < fstp:
        chunk = np.concatenate([chunk, np.repeat(chunk[-1:], fstp - len(chunk), axis=0)])
    return chunk[:fstp]

action_failures = []
for ep_id in check_ep_ids:
    sorted_rows = sorted(episodes[ep_id], key=lambda r: r["step_pos"])
    sid = sorted_rows[0]["sample_idx"]
    vp, pp = video_path(sid), parquet_path(sid)
    if not vp.exists() or not pp.exists():
        print(f"  SKIP ep {ep_id}: source files not on this node")
        continue

    vr = VideoReader(str(vp), num_threads=1, ctx=cpu(0))
    vfps, n_vid = vr.get_avg_fps(), len(vr)
    del vr
    fstp = max(1, ceil(vfps / TARGET_FPS))

    df = pd.read_parquet(pp, columns=["action"])
    raw_act = np.asarray(df["action"].to_list(), dtype=np.float32)[:, :ACTION_DIM]
    max_len = min(n_vid, len(raw_act))

    for row in sorted_rows:
        fi = row["frame_index"]
        expected = reconstruct_action_chunk(raw_act, fi, fstp, max_len).reshape(ACTIONS_PER_ROW)
        if not np.allclose(expected, row["actions"], rtol=1e-3, atol=1e-4):
            action_failures.append(
                f"ep {ep_id} step_pos={row['step_pos']} fi={fi}: "
                f"max_diff={np.max(np.abs(expected - row['actions'])):.2e}"
            )
    print(f"  ep {ep_id}: {len(sorted_rows)} action chunks — OK")

if action_failures:
    for f in action_failures[:5]: print("FAIL:", f)
    raise AssertionError(f"{len(action_failures)} action aggregation failures")
print("PASS Test 6 — actions == raw_actions[fi:fi+fstp, :ACTION_DIM].flatten()")

---
## Test 7 — Global Statistics & Coverage Estimate

Reports dataset-wide metrics from `index.json` (no data read) and a task-coverage
estimate from the sampled rows compared to the manifest.

In [ ]:
n_manifest = len(manifest["episodes"])
manifest_tasks = {ep["task_name"] for ep in manifest["episodes"]}

ep_lengths = {ep_id: ep_rows[0]["episode_len"] for ep_id, ep_rows in episodes.items()}
mean_ep_len = float(np.mean(list(ep_lengths.values()))) if ep_lengths else 0.0

estimated_duration_h = total_rows / TARGET_FPS / 3600
fraction_scanned     = n_subset / total_rows if total_rows > 0 else 0
estimated_total_eps  = len(episodes) / fraction_scanned if fraction_scanned > 0 else 0

sample_task_names = {
    manifest["episodes"][r["sample_idx"]]["task_name"]
    for r in rows_meta if r["sample_idx"] < n_manifest
}

print("=== Dataset Statistics (from index.json) ===")
print(f"  Total MDS rows        : {total_rows:>12,}  (one row = one sampled frame)")
print(f"  Total shards          : {len(shards):>12,}")
print(f"  Estimated duration    : {estimated_duration_h:>11.1f} h  (@ {TARGET_FPS} fps)")
print()
print(f"=== Manifest ===")
print(f"  Episodes in manifest  : {n_manifest:>12,}")
print(f"  Tasks in manifest     : {len(manifest_tasks):>12,}")
print()
print(f"=== Sample Coverage ({n_subset:,} rows, {len(sampled_shards)} shards, "
      f"{100*fraction_scanned:.2f}% of rows) ===")
print(f"  Unique episodes       : {len(episodes):>12,}")
print(f"  Mean episode length   : {mean_ep_len:>11.1f} frames")
print(f"  Estimated total eps   : {estimated_total_eps:>11.0f}")
print(f"  Tasks in sample       : {len(sample_task_names):>12,} / {len(manifest_tasks)}")
print()
print("PASS Test 7")

---
## Summary

| Test | What it checks | Scope |
|------|---------------|-------|
| 1 Schema | All MDS columns present in every shard | Full `index.json` (instant) |
| 2 Shape / Finiteness | Token shapes consistent; tokens, actions, states all finite (no NaN/Inf) | 5 representative shards (~1 min) |
| 3 Episode completeness | `step_pos 0…episode_len-1` no gaps; `frame_index[step_pos=0] == 0` | Episodes in sampled shards |
| 4 Frame-index stride | `frame_index` increases by `fstp` | Episodes in sampled shards |
| 5 State alignment | `states == extract_proprio(parquet[frame_index])` | `N_SPOT_CHECK` episodes |
| 6 Action aggregation | `actions == raw_actions[fi:fi+fstp].flatten()` | `N_SPOT_CHECK` episodes |
| 7 Coverage / stats | Row/shard counts, duration estimate, task coverage | Full `index.json` + sample |

**Remaining gaps:** pixel correctness; encoder output quality; episodes in unsampled shards.

---
## HF Upload — `huggingface-cli upload-large-folder`

Uses the CLI's `upload-large-folder` subcommand, which is designed for multi-TB datasets:

- **Chunked LFS uploads** with automatic retry per chunk
- **Resume**: re-running the same command skips already-uploaded chunks (tracked in a local `.cache` file)
- **`--num-workers`** controls parallel upload threads (16 is a good starting point on RunPod)
- `HF_HUB_ENABLE_HF_TRANSFER=1` enables the Rust-backed S3 uploader for 2–5× speed

Run the cell below **only after all tests pass**.

> Estimated time for 2.5 TB at 1 Gbit/s ≈ 5–6 h.  
> Safe to interrupt and re-run — progress is tracked automatically.

In [ ]:
import subprocess, os, shutil

# Redirect cache to workspace
os.makedirs("/workspace/.cache/huggingface", exist_ok=True)
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_TOKEN"] = "REDACTED_HF_TOKEN"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# Verify space
total, used, free = shutil.disk_usage("/")
print(f"Root free: {free/1e9:.1f} GB")

# Launch
cmd = [
    "hf", "upload-large-folder",
    "quastAI/behavior-1k-2025-challenge-vjepa2-vitg-demo-embeddings",
    "/workspace/shards_256px_vit_16_g",
    "--repo-type", "dataset",
    "--num-workers", "4",
]

log = open("/workspace/hf_upload.log", "a")
proc = subprocess.Popen(cmd, env=os.environ.copy(), stdout=log, stderr=log)
print(f"PID: {proc.pid}")